In [25]:
import warnings
import pandas as pd
import os


# Suppress all FutureWarnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Backtest trades function
def backtest_trades(price_data, signal_data, tp=None, sl=None, entry_time_offset=None, percentage_change=None, time_limit_minutes=None):
    output_data = pd.DataFrame(columns=[
        'Datetime', 'Side', 'Signal Open Price', 'Entry Price', 'TP Price', 'SL Price', 'Result', 'Duration', 'Execution Latency', 'ROI', 'NAV', 'Ignore Reason'
    ])
    
    initial_margin = 100000
    current_margin = initial_margin
    exit_datetimes = []

    for i, row in signal_data.iterrows():
        signal_datetime = row['Datetime']
        signal_value = row['Signal']
        
        if signal_value == 0:
            continue
        elif signal_value > 0:
            side = 'Buy'
        else:
            side = 'Sell'
        
        adjusted_signal_datetime = signal_datetime + pd.Timedelta(minutes=entry_time_offset)
        if adjusted_signal_datetime not in price_data.index:
            continue

        signal_open_price = price_data.at[adjusted_signal_datetime, 'Open']
        
        exit_datetimes.sort(key=lambda x: x[0])
        ignore_signal = False
        reason = ''
        if exit_datetimes:
            later_exits = [ed for ed in exit_datetimes if ed[0] > signal_datetime]

            if len(later_exits) >= 3:
                result = 'Ignored'
                reason = 'More than 2 open trades'
                ignore_signal = True
            elif len(later_exits) == 2:
                if later_exits[-1][1] == side:
                    ignore_signal = False
                else:
                    result = 'Ignored'
                    reason = 'Two open trades, last one with different side'
                    ignore_signal = True
            elif len(later_exits) == 1:
                if later_exits[-1][1] != side:
                    ignore_signal = False
                else:
                    result = 'Ignored'
                    reason = 'One open trade with the same side'
                    ignore_signal = True

        if ignore_signal:
            new_row = pd.DataFrame([{
                'Datetime': signal_datetime,
                'Side': side,
                'Signal Open Price': signal_open_price,
                'Entry Price': None,
                'TP Price': None,
                'SL Price': None,
                'Result': result,
                'Duration': '00:00:00',
                'Execution Latency': '00:00:00',
                'ROI': 0,
                'NAV': current_margin,
                'Ignore Reason': reason
            }])
            output_data = pd.concat([output_data, new_row], ignore_index=True)
            continue
        
        entry_datetime, entry_price, entry_duration = determine_entry(price_data, signal_datetime, percentage_change, side, time_limit_minutes, entry_time_offset)
        if entry_datetime is None:
            new_row = pd.DataFrame([{
                'Datetime': signal_datetime,
                'Side': side,
                'Signal Open Price': signal_open_price,
                'Entry Price': None,
                'TP Price': None,
                'SL Price': None,
                'Result': 'Not Filled',
                'Duration': '00:00:00',
                'Execution Latency': '00:00:00',
                'ROI': 0,
                'NAV': current_margin,
                'Ignore Reason': ''
            }])
            output_data = pd.concat([output_data, new_row], ignore_index=True)
            continue
        
        if side == 'Buy':
            tp_price = entry_price * (1 + tp)
            sl_price = entry_price * (1 - sl)
        else:
            tp_price = entry_price * (1 - tp)
            sl_price = entry_price * (1 + sl)
        
        result, duration_str = check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side)
        exit_datetime = entry_datetime + pd.Timedelta(duration_str)

        if result in [1, -1]:
            exit_datetimes.append((exit_datetime, side))

        if result == 1:
            current_margin = current_margin * (1 + tp)
        elif result == -1:
            current_margin = current_margin * (1 - sl)
        
        roi = ((current_margin - initial_margin) / initial_margin) * 100
        nav = current_margin
        initial_margin = current_margin
        
        new_row = pd.DataFrame([{
            'Datetime': signal_datetime,
            'Side': side,
            'Signal Open Price': signal_open_price,
            'Entry Price': entry_price,
            'TP Price': tp_price,
            'SL Price': sl_price,
            'Result': result,
            'Duration': duration_str,
            'Execution Latency': format_duration(entry_duration),
            'ROI': roi,
            'NAV': nav,
            'Ignore Reason': ''
        }])
        
        output_data = pd.concat([output_data, new_row], ignore_index=True)
    
    return output_data

# Helper functions
def format_duration(duration):
    seconds = duration.total_seconds()
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    seconds = int(seconds % 60)
    return f"{hours:02}:{minutes:02}:{seconds:02}"

def check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side):
    result = 0
    exit_datetime = None
    subsequent_prices = price_data.loc[entry_datetime:]

    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy':
            if price_row['High'] >= tp_price:
                result = 1
                exit_datetime = current_datetime
                break
            elif price_row['Low'] <= sl_price:
                result = -1
                exit_datetime = current_datetime
                break
        else:
            if price_row['Low'] <= tp_price:
                result = 1
                exit_datetime = current_datetime
                break
            elif price_row['High'] >= sl_price:
                result = -1
                exit_datetime = current_datetime
                break
    
    if exit_datetime:
        duration = exit_datetime - entry_datetime
        duration_str = format_duration(duration)
    else:
        duration_str = '00:00:00'
        
    return result, duration_str

def determine_entry(price_data, signal_datetime, percentage_change, side, time_limit_minutes, entry_time_offset):
    adjusted_signal_datetime = signal_datetime + pd.Timedelta(minutes=entry_time_offset)
    if adjusted_signal_datetime not in price_data.index:
        return None, None, None
    
    adjusted_open_price = price_data.at[adjusted_signal_datetime, 'Open']
    percentage_change_price = adjusted_open_price * (1 - percentage_change) if side == 'Buy' else adjusted_open_price * (1 + percentage_change)
    
    time_limit = adjusted_signal_datetime + pd.Timedelta(minutes=time_limit_minutes)
    subsequent_prices = price_data.loc[adjusted_signal_datetime:time_limit]
    
    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy' and price_row['Low'] <= percentage_change_price:
            entry_price = percentage_change_price
            duration = current_datetime - adjusted_signal_datetime
            return current_datetime, entry_price, duration
        elif side == 'Sell' and price_row['High'] >= percentage_change_price:
            entry_price = percentage_change_price
            duration = current_datetime - adjusted_signal_datetime
            return current_datetime, entry_price, duration

    return None, None, None

def calculate_metrics(group, initial_nav):
    total_trades = len(group[(group['Result'] == 1) | (group['Result'] == -1)])
    total_wins = len(group[group['Result'] == 1])
    total_losses = len(group[group['Result'] == -1])
    win_rate = total_wins / total_trades if total_trades > 0 else 0
    
    final_nav = group['NAV'].iloc[-1] if total_trades > 0 else initial_nav
    roi = ((final_nav - initial_nav) / initial_nav) * 100
    
    drawdown = 0
    cumulative_returns = (group['NAV'] - initial_nav).cumsum()
    peak = cumulative_returns.cummax()
    drawdown = (peak - cumulative_returns).max()
    
    return {
        'Total Trades': total_trades,
        'Total Wins': total_wins,
        'Total Losses': total_losses,
        'Win Rate': win_rate,
        'ROI': roi,
        'NAV': final_nav,
        'Max Drawdown': drawdown
    }

def generate_report(price_data, signal_data, scenarios, output_directory):
    report_columns = ['Scenario', 'Period', 'Total Trades', 'Total Wins', 'Total Losses', 'Win Rate', 'ROI', 'NAV', 'Max Drawdown']
    report_data = pd.DataFrame(columns=report_columns)

    for scenario in scenarios:
        tp = scenario['tp']
        sl = scenario['sl']
        percentage_change = scenario.get('percentage_change', None)
        entry_time_offset = scenario.get('entry_time_offset', None)
        time_limit_minutes = scenario.get('time_limit_minutes', None)
        
        # Backtest the trades for the current scenario
        trade_data = backtest_trades(price_data, signal_data, tp, sl, entry_time_offset, percentage_change, time_limit_minutes)
        monthly_groups = trade_data.groupby(trade_data['Datetime'].dt.to_period('M'))
        monthly_reports = []
        initial_nav = 100000
        for month, group in monthly_groups:
            metrics = calculate_metrics(group, initial_nav)
            metrics['Period'] = month.strftime('%Y-%m')
            metrics[
                'Scenario'] = f"TP={tp}, SL={sl}, Offset={entry_time_offset}, pctChange={percentage_change}, TimeLimit={time_limit_minutes}"
            monthly_reports.append(pd.DataFrame([metrics]))
            initial_nav = metrics['NAV']

        if monthly_reports:
            monthly_report = pd.concat(monthly_reports, ignore_index=True)
            report_data = pd.concat([report_data, monthly_report], ignore_index=True)

        overall_metrics = calculate_metrics(trade_data, 100000)
        overall_metrics['Period'] = 'Overall'
        overall_metrics[
            'Scenario'] = f"TP={tp}, SL={sl}, Offset={entry_time_offset}, pctChange={percentage_change}, TimeLimit={time_limit_minutes}"
        overall_report = pd.DataFrame([overall_metrics])
        report_data = pd.concat([report_data, overall_report], ignore_index=True)

    os.makedirs(output_directory, exist_ok=True)
    report_data.to_csv(os.path.join(output_directory, 'tow_backtesting.csv'), index=False)

    return report_data

In [15]:
price_data = pd.read_csv('E:\Signal Backtesting\Input\Price_2024.csv', parse_dates=['Datetime'],
                         index_col='Datetime')
# Re-load the signal data without setting the index
signal_data = pd.read_csv('E:\Signal Backtesting\Input\Signature_AI_Results_Final.csv', parse_dates=['Datetime'])

In [17]:
# Define the parameters
tp = 0.0179
sl = 0.0132
entry_time_offset = 183  # Time offset in minutes
percentage_change = 0.0028
time_limit_minutes = 120

# Call the backtest_trades function
backtest_output = backtest_trades(
    price_data=price_data,
    signal_data=signal_data,
    tp=tp,
    sl=sl,
    entry_time_offset=entry_time_offset,
    percentage_change=percentage_change,
    time_limit_minutes=time_limit_minutes
)

backtest_output

,Datetime,Side,Signal Open Price,Entry Price,TP Price,SL Price,Result,Duration,Execution Latency,ROI,NAV,Ignore Reason
0,2024-01-01 00:00:00,Sell,42627.9,NaN,NaN,NaN,Not Filled,00:00:00,00:00:00,0,100000,
1,2024-01-01 16:00:00,Buy,43166.5,NaN,NaN,NaN,Not Filled,00:00:00,00:00:00,0,100000,
2,2024-01-02 00:00:00,Sell,45283.8,45410.59464,44597.744996,46010.014489,1,31:59:00,00:22:00,1.79,101790.0,
3,2024-01-02 22:00:00,Buy,45183.7,NaN,NaN,NaN,Not Filled,00:00:00,00:00:00,0,101790.0,
4,2024-01-03 16:00:00,Buy,42242.0,NaN,NaN,NaN,Not Filled,00:00:00,00:00:00,0,101790.0,
...,...,...,...,...,...,...,...,...,...,...,...,...
214,2024-06-27 08:00:00,Sell,61141.5,61312.69620,60215.198938,62122.023790,-1,02:12:00,00:46:00,-1.32,119013.296291,
215,2024-06-28 08:00:00,Buy,61528.8,61356.51936,62454.801057,60546.613304,-1,06:47:00,01:27:00,-1.32,117442.32078,
216,2024-06-28 16:00:00,Sell,60773.8,NaN,NaN,NaN,Not Filled,00:00:00,00:00:00,0,117442.32078,
217,2024-06-29 03:00:00,Buy,60794.7,NaN,NaN,NaN,Not Filled,00:00:00,00:00:00,0,117442.32078,


In [27]:
senario=[
{ 'tp': 0.0179,'sl' : 0.0132,'entry_time_offset':183,
'percentage_change' : 0.0028,
'time_limit_minutes' : 120}
]
output_directory ='E:\Signal Backtesting\Output'
report=generate_report(price_data, signal_data, scenarios=senario, output_directory=output_directory)
report

,Scenario,Period,Total Trades,Total Wins,Total Losses,Win Rate,ROI,NAV,Max Drawdown
0,"TP=0.0179, SL=0.0132, Offset=183, pctChange=0....",2024-01,21,16,5,0.761905,24.287523,124287.523325,0.000000
1,"TP=0.0179, SL=0.0132, Offset=183, pctChange=0....",2024-02,11,4,7,0.363636,-2.180724,121577.155215,17289.254773
2,"TP=0.0179, SL=0.0132, Offset=183, pctChange=0....",2024-03,27,10,17,0.370370,-4.732112,115823.988098,273452.335971
3,"TP=0.0179, SL=0.0132, Offset=183, pctChange=0....",2024-04,22,9,13,0.409091,-1.298248,114320.305030,64754.226768
4,"TP=0.0179, SL=0.0132, Offset=183, pctChange=0....",2024-05,16,8,8,0.500000,3.627266,118467.006900,9860.882448
5,"TP=0.0179, SL=0.0132, Offset=183, pctChange=0....",2024-06,18,7,11,0.388889,-2.173537,115892.082146,5648.983111
6,"TP=0.0179, SL=0.0132, Offset=183, pctChange=0....",Overall,115,54,61,0.469565,15.892082,115892.082146,0.000000


In [28]:
senario=[
{ 'tp': 0.0146,'sl' : 0.0143,'entry_time_offset':102,
'percentage_change' : 0.0,
'time_limit_minutes' : 120}
]
output_directory ='E:\Signal Backtesting\Output'
report=generate_report(price_data, signal_data, scenarios=senario, output_directory=output_directory)
report

,Scenario,Period,Total Trades,Total Wins,Total Losses,Win Rate,ROI,NAV,Max Drawdown
0,"TP=0.0146, SL=0.0143, Offset=102, pctChange=0....",2024-01,34,17,17,0.500000,0.155187,100155.187218,14055.064576
1,"TP=0.0146, SL=0.0143, Offset=102, pctChange=0....",2024-02,27,19,8,0.703704,17.370878,117553.022796,0.000000
2,"TP=0.0146, SL=0.0143, Offset=102, pctChange=0....",2024-03,39,18,21,0.461538,-4.071571,112766.767785,323722.316173
3,"TP=0.0146, SL=0.0143, Offset=102, pctChange=0....",2024-04,36,17,19,0.472222,-2.688770,109734.728309,198806.589232
4,"TP=0.0146, SL=0.0143, Offset=102, pctChange=0....",2024-05,33,18,15,0.545455,4.587262,114768.547345,13565.177203
5,"TP=0.0146, SL=0.0143, Offset=102, pctChange=0....",2024-06,31,16,15,0.516129,1.598916,116603.600471,1630.870750
6,"TP=0.0146, SL=0.0143, Offset=102, pctChange=0....",Overall,200,105,95,0.525000,16.603600,116603.600471,14055.064576


In [29]:
senario=[
{ 'tp': 0.0085,'sl' : 0.0186,'entry_time_offset':204,
'percentage_change' : 0.0023,
'time_limit_minutes' : 120}
]
output_directory ='E:\Signal Backtesting\Output'
report=generate_report(price_data, signal_data, scenarios=senario, output_directory=output_directory)
report

,Scenario,Period,Total Trades,Total Wins,Total Losses,Win Rate,ROI,NAV,Max Drawdown
0,"TP=0.0085, SL=0.0186, Offset=204, pctChange=0....",2024-01,23,18,5,0.782609,6.022127,106022.127335,0.000000
1,"TP=0.0085, SL=0.0186, Offset=204, pctChange=0....",2024-02,16,11,5,0.687500,-0.077062,105940.424880,2916.293437
2,"TP=0.0085, SL=0.0186, Offset=204, pctChange=0....",2024-03,30,26,4,0.866667,15.599959,122467.087949,0.000000
3,"TP=0.0085, SL=0.0186, Offset=204, pctChange=0....",2024-04,24,18,6,0.750000,4.050116,127427.146788,21813.237618
4,"TP=0.0085, SL=0.0186, Offset=204, pctChange=0....",2024-05,19,13,6,0.684211,-0.261449,127093.989933,57497.932486
5,"TP=0.0085, SL=0.0186, Offset=204, pctChange=0....",2024-06,23,16,7,0.695652,0.400717,127603.277747,1538.268616
6,"TP=0.0085, SL=0.0186, Offset=204, pctChange=0....",Overall,135,102,33,0.755556,27.603278,127603.277747,0.000000
